In [ ]:
import pandas as pd
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
import requests

In [ ]:
# load.env-file
load_dotenv()

schema = "sql_cities"
host = "127.0.0.1"
user = "root"
password = os.getenv("DB_PASSWORD")
port = 3306

connection_string = f'mysql+pymysql://{user}:{password}@{host}:{port}/{schema}'

In [ ]:
url = "https://aerodatabox.p.rapidapi.com/airports/search/location"

querystring = {"lat":"52.31","lon":"13.24","radiusKm":"50","limit":"50","withFlightInfoOnly":"true"}

headers = {
    "Accept": "application/json, application/xml",
    "X-RapidAPI-Key": "x_rapid_api_key",
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

response = requests.get(url, headers=headers, params=querystring)

print(response.json())

In [ ]:
code = "EDDB"
tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")

time_windows = [
    (f"{tomorrow}T00:00", f"{tomorrow}T12:00"),
    (f"{tomorrow}T12:00", f"{tomorrow}T23:59")
]

headers = {
    "Accept": "application/json",
    "X-RapidAPI-Key": "x_rapid_api_key",
    "X-RapidAPI-Host": "aerodatabox.p.rapidapi.com"
}

params = {
    "direction": "Arrival",
    "withLeg": "false",
    "withCancelled": "false",
    "withCodeshared": "true",
    "withCargo": "false",
    "withPrivate": "false",
    "withLocation": "true"
}

all_flights = []

for fromLocal, toLocal in time_windows:
    url = f"https://aerodatabox.p.rapidapi.com/flights/airports/icao/{code}/{fromLocal}/{toLocal}"
    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    result = response.json()
    all_flights.extend(result if isinstance(result, list) else result.get("arrivals", []))

print(f"arrivals on {tomorrow}:")
print(all_flights)

In [ ]:
#flight_data_df = pd.json_normalize(response.json()["arrivals"])

In [ ]:
#pd.json_normalize(response.json()["arrivals"])
cities_flight_data_df = pd.json_normalize(response.json()["arrivals"])

cities_flight_data_df

In [ ]:
#cities_flight_data_df.to_sql('cities_flight_data',
 #                 if_exists='append',
  #                con=connection_string,
   #               index=False)